# Reproduction of Naresh & Dullam (2026), *European Transport Research Review* 18:59

This notebook regenerates **every table and figure** of the article from the public STATS19 files and runs the **live decision-support system** (GPS → live weather → Google Routes → risk scoring → risk-aware route → SMS alert).

* All numbers are computed at run time; nothing is copied from the article.
* `outputs/CLAIMS_AUDIT.md` lists each claim of the article next to the value this code produces.
* The live services (GPS, Google Routes, Twilio, weather) depend on time, place and network, so their timings differ from run to run. Every run is logged, and Tables 11–13 are computed from that log.

**Runtime:** about 1.5–3 h on a standard Colab CPU for the full paper (the random forests and the 5-fold CV dominate). `--fast` gives a ~20-minute check.

## 1. Setup

In [ ]:
REPO_URL = "https://github.com/vsnaresh111/accident-classification"   # repository holding this code
import os, subprocess
if not os.path.exists("rsr"):
    if not os.path.exists("repo"):
        subprocess.run(["git", "clone", "-q", REPO_URL, "repo"], check=True)
    os.chdir("repo")
!pip -q install -r requirements.txt
print(os.getcwd())

In [ ]:
# Source data: the three public STATS19 files (release 'dataset')
import os
os.makedirs("data", exist_ok=True)
base = "https://github.com/vsnaresh111/accident-classification/releases/download/dataset/"
for f in ("accidents.csv", "vehicles.csv", "casualties.csv"):
    if not os.path.exists(f"data/{f}"):
        !wget -q -O data/{f} {base}{f}
!ls -la data
os.environ["ACCIDENT_DATA_DIR"] = "data"
os.environ["RSR_OUTPUT_DIR"] = "outputs"

In [ ]:
# Credentials: add them in Colab (key icon -> Secrets). Values are never printed.
import os
try:
    from google.colab import userdata
    for k in ("GOOGLE_MAPS_API_KEY", "TWILIO_ACCOUNT_SID", "TWILIO_AUTH_TOKEN",
              "TWILIO_PHONE_NUMBER", "EMERGENCY_PHONE_NUMBER"):
        try:
            os.environ[k] = userdata.get(k) or ""
        except Exception:
            os.environ.setdefault(k, "")
except ImportError:
    pass
print({k: bool(os.environ.get(k)) for k in ("GOOGLE_MAPS_API_KEY", "TWILIO_ACCOUNT_SID",
      "TWILIO_AUTH_TOKEN", "TWILIO_PHONE_NUMBER", "EMERGENCY_PHONE_NUMBER")})

## 2. Reproduce the paper (Tables 4–10, 15; Figs 3–6, 8, 10, 12, 13)

`--graph osm` downloads the OpenStreetMap drive network around `--center` for the routing evaluation (Table 9, Fig. 10). Route benefit is measured against **held-out** accidents (the test split), so it is not circular.

In [ ]:
!python run_paper.py --graph osm --graph-cache outputs/cache/osm_london.graphml --center "51.5074,-0.1278" --radius-m 6000 --od-pairs 300 --export-dataset
# optional: Table 9 for one named pair -> add  --case-origin "LAT,LON" --case-dest "LAT,LON"
!python leakage_diagnostic_pipeline.py   # negative control (concern 6)
# faster check:  !python run_paper.py --fast --graph osm

In [ ]:
from IPython.display import Markdown, display, Image
display(Markdown(open("outputs/CLAIMS_AUDIT.md").read()))

In [ ]:
import pandas as pd, glob
for f in sorted(glob.glob("outputs/tables/table*.csv")):
    print("\n" + f); display(pd.read_csv(f))

In [ ]:
for f in sorted(glob.glob("outputs/figures/*.png")):
    print(f); display(Image(f, width=700))

## 3. Live system

The risk model, hotspot grid and threshold are loaded from `outputs/models/system_bundle.joblib`.
`dry_run=True` computes and logs the alert decision without sending an SMS; `dry_run=False` sends real messages.
Location comes from the browser's GPS (allow location access when asked), else IP geolocation, else the manual `origin`.
Risk is available only inside Great Britain (the coverage of the accident data); elsewhere the system says so and never raises an alert.

In [ ]:
import sys; sys.path.insert(0, ".")
from rsr import live
from IPython.display import IFrame
system = live.RiskSystem("outputs/models/system_bundle.joblib")
print(system.meta["model"], "| alert score:", system.meta["alert_score"], "| tau =", system.tau)
profile = {**system.meta["default_profile"], "Age_of_Driver": 30, "Sex_of_Driver": "1"}

In [ ]:
# One trip from your current location (browser GPS) to a destination
res = live.run_trip(system, destination="London Bridge Station, London", gps_mode="auto",
                    profile=profile, dry_run=True,
                    log_path="outputs/logs/system_logs.csv", map_path="outputs/route_map.html")
IFrame("outputs/route_map.html", width=900, height=520)

In [ ]:
# The same trip from a fixed origin (reproducible)
res = live.run_trip(system, destination=(51.5033, -0.1196), origin=(51.5155, -0.0922),
                    gps_mode="manual", profile=profile, dry_run=True,
                    log_path="outputs/logs/system_logs.csv", map_path="outputs/route_map.html")
IFrame("outputs/route_map.html", width=900, height=520)

In [ ]:
# Real-time tracking: re-locate and re-evaluate every 20 s (5 updates)
runs = live.track(system, (51.5033, -0.1196), updates=5, interval_s=20, gps_mode="auto",
                  profile=profile, dry_run=True, log_path="outputs/logs/system_logs.csv")

In [ ]:
# SMS: send ONE clearly labelled test message and wait for the delivery receipt
loc = live.get_location("auto")
sms = live.send_sms(live.alert_body(loc["lat"], loc["lon"], 1.0, "system test", test=True), dry_run=False)
print(sms)

### Tables 11–13 from repeated live runs
Runs the full pipeline for origin–destination pairs across London and computes the tables from the log. Set `SEND_SMS = True` to send real SMS for runs that cross the threshold.

In [ ]:
import numpy as np, pandas as pd
SEND_SMS = False
rng = np.random.default_rng(0)
pairs = [((51.50 + rng.uniform(-0.05, 0.05), -0.12 + rng.uniform(-0.08, 0.08)),
          (51.50 + rng.uniform(-0.05, 0.05), -0.12 + rng.uniform(-0.08, 0.08))) for _ in range(30)]
for o, d in pairs:
    live.run_trip(system, d, origin=o, gps_mode="manual", profile=profile, dry_run=not SEND_SMS,
                  log_path="outputs/logs/system_logs.csv", verbose=False)
L = pd.read_csv("outputs/logs/system_logs.csv")
t11, t13 = live.table11(L), live.table13(L)
t12 = live.table12(system, res["routes"], res["conditions"])
display(t11); display(t12); display(t13)
t11.to_csv("outputs/tables/table11_realtime_performance.csv", index=False)
t12.to_csv("outputs/tables/table12_scalability.csv", index=False)
t13.to_csv("outputs/tables/table13_reliability.csv", index=False)

In [ ]:
# Offline tests of the system logic (mocked services)
!RSR_TEST_BUNDLE=outputs/models/system_bundle.joblib python -m pytest -q tests

In [ ]:
# Package every output for deposit
!zip -qr outputs.zip outputs -x "outputs/analytical_dataset.csv.gz" "outputs/cache/*/*"
from google.colab import files; files.download("outputs.zip")